### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [ ]:
!pip install langchain

In [18]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [ ]:
!pip install langchain_groq

In [19]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002400336B670>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002400336A500>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [20]:
!pip install langchain_core

In [21]:
from langchain_core.messages import HumanMessage,SystemMessage

In [22]:
#from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [23]:
result

AIMessage(content='Bonjour Comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 47, 'total_tokens': 54, 'completion_time': 0.022287962, 'completion_tokens_details': None, 'prompt_time': 0.010147313, 'prompt_tokens_details': None, 'queue_time': 0.04576371, 'total_time': 0.032435275}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9a56-6742-7732-88ea-27575d85fd81-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 7, 'total_tokens': 54})

In [25]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour Comment allez-vous ?'

In [24]:
StrOutputParser().invoke(result)

'Bonjour Comment allez-vous ?'

In [26]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour Comment allez-vous ?'

In [28]:
chain

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002400336B670>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002400336A500>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

In [27]:
model|parser.invoke(messages)

ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value=[SystemMessage(content='T..., response_metadata={})], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [ ]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [ ]:
prompt1=ChatPromptTemplate.from_messages(
    [("system","Translate the following into {language}:"),("user","{text}")]
)

In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([

    ("system", "Translate the following into {language}:"),
    ("user", "{text}")
]
)

In [ ]:
prompt.invoke({"language":"French","text":"Hello How are you?"}).to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello How are you?', additional_kwargs={}, response_metadata={})]

In [34]:

from langchain_core.prompts import ChatPromptTemplate

result=prompt.invoke({"language":"French","text":"Hello"})

In [35]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [36]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour.'

In [ ]:
!pip install streamlit

In [ ]:
 ###Added text at last line

In [37]:
prompt1=ChatPromptTemplate.from_messages(
    [("system","Translate the following into {language}:"),("user","{text}")]
)

In [38]:
prompt1.invoke({"language":"French","text":"Hello How are you?"}).to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello How are you?', additional_kwargs={}, response_metadata={})]

In [40]:
chain1 = prompt1|model|parser

print("Telugu:", chain1.invoke({"language":"Telugu","text":"Hello How are you?"}))

print("Spanish:", chain1.invoke({"language":"Spanish","text":"Hello How are you?"}))

print("French:", chain1.invoke({"language":"French","text":"Hello How are you?"}))

Telugu: నమస్కారం! నేను ఏం చేస్తున్నా? ధన్యవాదాలు అందించబడిన ప్రశ్నకు నాకు భాగ్యంగా ఉంది. నేను మంచిది. మీరు ఎలా ఉన్నారు?
Spanish: "Hola, ¿cómo estás?"
French: Bonjour Comment allez-vous?
